In [2]:
# Explore nanopore data

In [3]:
import polars as pl

In [23]:
df = (
    pl.scan_csv('../nanopore-dl/mag+gtdb.x.mf.manysearch.csv')
    .filter(pl.col('intersect_hashes') >= 3)
).collect()

In [24]:
df

query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""GCA_963606895 s__Methanocatell…","""98787da9033ca08f4a8e0e553cd3be…",null,0.001986,5,21,1000,"""DNA""","""8fd570c89a8d877373d6f73b129184…",0.000075,0.001986,1.0,1.0,0.0,0.743583,0.637509,0.690546,0.743583,5,84774
"""GCA_030461285 s__Tetragenococc…","""e97908effdb63763b179b45d8fc801…",null,0.001804,3,21,1000,"""DNA""","""8fd570c89a8d877373d6f73b129184…",0.000046,0.001804,1.0,1.0,0.0,0.740192,0.622189,0.68119,0.740192,3,84774
"""GCF_003238525 s__Staphylococcu…","""11ffa81e75a4bb46b35df6c7051cd1…",null,0.000158,3,21,1000,"""DNA""","""8fd570c89a8d877373d6f73b129184…",0.000036,0.000158,1.333333,1.0,0.471405,0.659052,0.622189,0.64062,0.659052,4,84774
"""GCA_963608965 s__Alloprevotell…","""19b86708cce7d31b96c20787494b5a…",null,0.001875,16,21,1000,"""DNA""","""8fd570c89a8d877373d6f73b129184…",0.000221,0.001875,1.0,1.0,0.0,0.741551,0.673815,0.707683,0.741551,16,84774
"""GCF_021376155 s__Enterococcus_…","""b3c5877c5d7a06e7545e36461376e2…",null,0.000636,5,21,1000,"""DNA""","""8fd570c89a8d877373d6f73b129184…",0.00007,0.000636,1.2,1.0,0.4,0.704332,0.637509,0.67092,0.704332,6,84774
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GCA_031974155 s__Chryseobacter…","""f8b97fc554470ae98e10b908dc3888…",null,0.001323,5,21,1000,"""DNA""","""b893cc2a564ec11c15aa5a42d4dcd1…",0.000009,0.001323,1.2,1.0,0.4,0.729345,0.575361,0.652353,0.729345,6,1335899
"""GCA_041281825 s__Paenibacillus…","""fa31ab55629366708d378c8293690b…",null,0.001009,3,21,1000,"""DNA""","""b893cc2a564ec11c15aa5a42d4dcd1…",0.000005,0.001009,2.0,1.0,1.414214,0.719996,0.561534,0.640765,0.719996,6,1335899
"""GCA_022782865 s__Ruminococcoid…","""1ecbf1b92aa9fa8d7f3ce1c01f584e…",null,0.001986,5,21,1000,"""DNA""","""b893cc2a564ec11c15aa5a42d4dcd1…",0.000009,0.001986,1.6,1.0,0.8,0.743583,0.575361,0.659472,0.743583,8,1335899


In [25]:
OLD_NAMES = [ x.strip() for x in open('../../2025-workflow-core99/inputs.cds/names.list') ]
OLD_NAMES_PLUS = [ x.strip() for x in open('../../2025-workflow-core99/inputs.cds/names-plus.list') ]

In [26]:
OLD_NAMES

['s__Bariatricus sp004560705',
 's__Colivicinus sp002299675',
 's__Cryptobacteroides sp000432655',
 's__Cryptobacteroides sp000434935',
 's__Cryptobacteroides sp034089285',
 's__Cryptobacteroides sp900546925',
 's__Fimisoma sp002320005',
 's__Floccifex porci',
 's__JAFBIX01 sp021531895',
 's__Lactobacillus amylovorus',
 's__Mogibacterium_A kristiansenii',
 's__Ornithospirochaeta sp022785155',
 's__Prevotella sp000434975',
 's__Prevotella sp002251295',
 's__Sodaliphilus sp004557565',
 's__UBA2868 sp004552595']

In [27]:
xx_df = (
    df
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).filter(pl.col('species_name').is_in(OLD_NAMES))


In [28]:
len(OLD_NAMES)

16

In [29]:
xx_df.group_by('match_md5').agg(pl.len())

match_md5,len
str,u32
"""16e675c20c1c0fe49a71975b78237d…",15
"""8fd570c89a8d877373d6f73b129184…",15
"""b893cc2a564ec11c15aa5a42d4dcd1…",16


In [13]:
xx_df = (
    by_species
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).filter((pl.col('species_name').is_in(OLD_NAMES_PLUS)))
with pl.Config(tbl_rows=-1):
    print(xx_df.sort('freq', descending=True))

NameError: name 'by_species' is not defined

In [13]:
len(OLD_NAMES_PLUS)

27

## Ask questions about core species in metagenomes by metagenome

In [14]:
by_acc = (
    df
    .with_columns(
        species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " ")
    )
    .filter(pl.col('species_name').is_in(OLD_NAMES))
    .group_by('match_name').agg(
        n_core=pl.len()
    )
).sort('n_core')


In [15]:
by_acc


match_name,n_core
str,u32
"""SRR12795729""",1
"""SRR12795774""",1
"""SRR12795782""",1
"""SRR12795740""",1
"""SRR12795783""",1
…,…
"""SRR11183573""",16
"""SRR11126323""",16
"""SRR17241519""",16


In [16]:
low_core_acc = by_acc.filter(pl.col('n_core') < 10)

In [17]:
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
#    .select(["acc", "organism", "bioproject", "mbases", "host", "project_name"]) 
    .collect()
)

In [18]:
low_core_acc = low_core_acc.rename({'match_name': 'acc'})

In [19]:
low_core_acc2 = low_core_acc.join(metadata_df, on='acc', how='left')
low_core_acc2

acc,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR12795729""",1,"""pig_colon_412""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396807""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264850""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-13-243200076""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,7340,2352,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494397""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""7340447032""","""2467034839""","""""2020-10-08T12:26:00.000Z""""",null,"""""16396807"""""
"""SRR12795774""",1,"""pig_colon_406""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396805""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264805""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-07-243200099""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,6253,1981,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494382""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""6253850126""","""2077501527""","""""2020-10-08T12:22:00.000Z""""",null,"""""16396805"""""
"""SRR12795782""",1,"""pig_colon_424""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396824""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264797""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-25-243200085""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,8907,2766,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494374""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""8907858440""","""2900407656""","""""2020-10-08T12:24:00.000Z""""",null,"""""16396824"""""
"""SRR12795740""",1,"""pig_colon_409""",[],"""WGS""",302,

In [20]:
low_core_acc2.group_by('bioproject').agg(
    pl.len()
).sort('len')

bioproject,len
str,u32
"""PRJNA741980""",1
"""PRJNA629856""",1
"""PRJNA471402""",1
"""PRJNA408025""",3
"""PRJEB31650""",5
"""PRJNA807368""",7
"""PRJNA373834""",8
"""PRJNA526405""",14
"""PRJNA668104""",17


## Explore saturation

In [21]:
sat_df = pl.read_parquet('../../2025-ccbaumler-wort-gathering/wort-sra-signature-saturation.parquet')

In [22]:
sat_df

filepath,signature,novel_count,total_count,sequence_saturation
str,str,i64,i64,f64
"""/group/ctbrowngrp/irber/data/w…","""SRR33867156""",23507,397539,0.940869
"""/group/ctbrowngrp/irber/data/w…","""ERR4022268""",27896,308529,0.909584
"""/group/ctbrowngrp/irber/data/w…","""ERR3771976""",223,3174,0.929742
"""/group/ctbrowngrp/irber/data/w…","""ERR11996469""",20,40,0.5
"""/group/ctbrowngrp/irber/data/w…","""ERR10755277""",3185,26976,0.881932
…,…,…,…,…
"""/group/ctbrowngrp/irber/data/w…","""SRR9934015""",57128,286525,0.800618
"""/group/ctbrowngrp/irber/data/w…","""SRR7588346""",67307,409191,0.835512
"""/group/ctbrowngrp/irber/data/w…","""SRR10432619""",88275,942001,0.90629


In [23]:
sat_df2 = df.join(sat_df, left_on='match_name', right_on='signature',
                  how='left')

In [24]:
zz = []
for min_sat in (0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    xx_df = sat_df2.filter(pl.col('sequence_saturation') >= min_sat)
    n_acc = xx_df['match_name'].n_unique()
    xx2_df= df.group_by('query_name').agg(
        freq=pl.len() / n_acc
    ).filter(pl.col('freq') >= 0.95)
    zz.append(dict(min=min_sat, num_metag=n_acc, num_core95=len(xx2_df)))
    #print(min_sat, n_acc, len(xx2_df))

zz_df = pl.DataFrame(zz)


In [25]:
zz_df

min,num_metag,num_core95
f64,i64,i64
0.25,3216,17
0.3,3215,17
0.4,3206,18
0.5,3065,38
0.6,2251,269
0.7,1244,737
0.8,366,1724
0.9,37,3116
